# 03 — Benchmarks : le score à battre

## Objectif

Avant de construire des modèles, je calcule deux **benchmarks** : des prévisions très simples,
**sans apprentissage**, qui recopient le passé de la consommation. Ils donnent **le score à
battre** : un modèle qui ne fait pas mieux qu'eux ne sert à rien.

Le sujet le demande : *« au moins deux références simples »* et *« un modèle complexe ne
présente un intérêt que si son gain par rapport à ces benchmarks est établi »*.

## Pourquoi je peux les faire dès maintenant

Un benchmark **n'apprend rien** : il recopie la consommation des semaines précédentes. Il n'a
besoin **que de la consommation** (prête depuis le notebook 01), ni de la météo ni du
calendrier. Les outils écrits ici (la façon de noter, la vérification de la règle des 14 h)
serviront ensuite tels quels pour les modèles.

## Les étapes de ce notebook

1. Ranger la consommation par jour et par heure de Paris
2. Les deux benchmarks : B1 et B2
3. Vérifier la règle des 14 h (aucune triche)
4. La façon de noter les prévisions
5. Les résultats sur l'apprentissage (2016-2022) et la validation (2023)
6. Où les benchmarks se trompent, et ce que ça nous apprend pour les modèles

**Le test 2024-2025 n'est pas utilisé ici** : il ne servira qu'une seule fois, à la fin, après
avoir gelé tous nos choix. Le code refuse d'ailleurs de le calculer.

## Où est le code ?

- `src/benchmarks.py` : les deux benchmarks et la vérification de la règle des 14 h ;
- `src/evaluation.py` : les mesures d'erreur, les mêmes pour toutes les méthodes ;
- `tests/test_benchmarks.py` et `tests/test_evaluation.py` : les tests automatiques.

Ce notebook appelle ces fonctions et explique les résultats. Relancer sans le notebook :
`python -m src.benchmarks`.

In [ ]:
import sys
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Je cherche la racine du projet (le dossier qui contient "src") pour pouvoir importer le code
RACINE = Path.cwd().resolve()
while not (RACINE / "src").exists() and RACINE != RACINE.parent:
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE))

from src import benchmarks, config, evaluation, rte

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


def voir_code(fonction):
    """Affiche le code d'une fonction de src/, pour le lire sans quitter le notebook."""
    print(inspect.getsource(fonction))


# Mêmes couleurs que le notebook 01 : la vraie consommation en encre foncée,
# B1 en bleu, B2 en orange (toujours les mêmes couleurs pour les mêmes méthodes).
REEL, BLEU, ORANGE = "#2b3440", "#2a78d6", "#eb6834"
GRIS_TEXTE = "#52514e"
plt.rcParams.update({
    "figure.figsize": (11, 4), "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": GRIS_TEXTE, "xtick.color": GRIS_TEXTE,
    "ytick.color": GRIS_TEXTE, "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.titleweight": "bold", "axes.titlesize": 12, "lines.linewidth": 1.5,
})

conso_h = rte.lire_prepare()  # le fichier préparé dans le notebook 01
print(f"{len(conso_h):,} heures de consommation chargées")

## 1. Ranger la consommation par jour et par heure de Paris

**Pourquoi :** le problème est posé en heure de Paris (« les 24 heures de demain »). Pour
recopier « mercredi dernier à 18 h », le plus simple est un tableau avec **une ligne par jour**
et **une colonne par heure** (0 à 23).

**Les jours de changement d'heure** n'ont pas 24 heures :
- dernier dimanche de mars : **23 h**, l'heure 2 n'existe pas → sa case reste vide ;
- dernier dimanche d'octobre : **25 h**, l'heure 2 existe deux fois → je fais la moyenne des deux.

In [ ]:
voir_code(benchmarks.consommation_par_jour)

In [ ]:
conso_jour = benchmarks.consommation_par_jour(conso_h)
print(f"{conso_jour.shape[0]} jours x {conso_jour.shape[1]} heures")

# Exemple : la semaine du passage à l'heure d'été 2023 (le dimanche 26 mars, l'heure 2 est vide)
(conso_jour.loc["2023-03-24":"2023-03-28", 0:5] / 1000).round(1)

## 2. Les deux benchmarks

On veut prévoir le jour cible J+1. Par exemple, le **mercredi 11 janvier 2023**, la prévision
étant faite le mardi 10 à 14 h.

| Benchmark | Règle | Pour le mercredi 11/01/2023 |
|---|---|---|
| **B1** « semaine précédente » | recopier, heure par heure, le même jour de la semaine précédente | le mercredi 4 janvier |
| **B2** « moyenne de 4 semaines » | moyenne, heure par heure, des 4 mêmes jours précédents | les mercredis 4 janvier, 28, 21 et 14 décembre |

**Pourquoi ces deux-là :** ce sont les deux références proposées par le sujet. B1 suit les
changements récents ; B2 est plus stable (un jour bizarre pèse 4 fois moins), mais réagit plus
lentement. On va voir lequel des deux gagne.

In [ ]:
voir_code(benchmarks.moyenne_des_jours_precedents)

In [ ]:
# La semaine du 16 au 22 janvier 2023 : une vague de froid arrive à partir du jeudi 19
jours = pd.date_range("2023-01-16", "2023-01-22", freq="D")
b1 = benchmarks.benchmark_b1(conso_jour, jours)
b2 = benchmarks.benchmark_b2(conso_jour, jours)
vrai = conso_jour.loc[jours]


def en_courbe(tableau):
    """Remet un tableau jour x heure en une seule courbe horaire (pour le graphique)."""
    serie = tableau.stack()
    serie.index = [jour + pd.Timedelta(hours=heure) for jour, heure in serie.index]
    return serie / 1000


fig, ax = plt.subplots(figsize=(12, 4.2))
ax.plot(en_courbe(vrai), color=REEL, linewidth=2, label="Consommation réelle")
ax.plot(en_courbe(b1), color=BLEU, label="B1 : semaine précédente")
ax.plot(en_courbe(b2), color=ORANGE, label="B2 : moyenne de 4 semaines")
ax.set_ylabel("GW")
ax.set_title("Prévisions des benchmarks, semaine du 16 janvier 2023")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)
plt.show()

**Ce que je lis :**
- Les deux benchmarks reproduisent bien la **forme** de la journée : creux la nuit, pointe le
  matin, pointe le soir.
- Mais le **niveau** est trop bas **toute la semaine**. La semaine précédente (9-15 janvier)
  était plus douce : en la recopiant, on sous-estime le chauffage.
- L'écart se creuse à partir du **jeudi 19**, avec l'arrivée d'une vague de froid, et dépasse
  15 GW le week-end. B2, qui regarde encore plus loin en arrière, est encore plus bas.

Les benchmarks ne peuvent pas voir venir le froid, puisqu'ils ne connaissent pas la
température. C'est exactement ce que les modèles avec météo devront corriger.

## 3. Vérifier la règle des 14 h

Le jour le plus récent utilisé par les benchmarks est **7 jours avant le jour cible**, donc
6 jours avant le jour J. Il est entièrement terminé à 14 h le jour J. Je ne me contente pas de
le dire : je le **vérifie** pour chaque jour, avec la règle de `src/protocole.py`.

In [ ]:
voir_code(benchmarks.verifier_disponibilite)

In [ ]:
jours_2023 = benchmarks.jours_de_la_periode("validation")
benchmarks.verifier_disponibilite(jours_2023, benchmarks.DECALAGE_B1)
benchmarks.verifier_disponibilite(jours_2023, benchmarks.DECALAGES_B2)
print("B1 et B2 : aucune heure inconnue à 14 h n'est utilisée.")

# Et la vérification sait détecter une triche : recopier la veille du jour cible
# (le jour J lui-même) est interdit, car à 14 h le soir du jour J n'a pas encore eu lieu.
try:
    benchmarks.verifier_disponibilite(pd.DatetimeIndex(["2023-01-11"]), [1])
except ValueError as erreur:
    print("Triche détectée :", erreur)

## 4. La façon de noter les prévisions

Pour chaque jour cible, je compare les 24 heures prévues aux 24 heures réelles. Le sujet
demande **plusieurs mesures complémentaires** :

| Mesure | Question | Unité |
|---|---|---|
| **MAE** | De combien se trompe-t-on en moyenne ? | MW |
| **RMSE** | Pareil, mais les grosses erreurs comptent plus | MW |
| **MAPE** | La MAE en pourcentage de la consommation | % |
| **Biais** | Prévoit-on trop (positif) ou pas assez (négatif) ? | MW |
| **Erreur énergie du jour** | A-t-on bien prévu le total de la journée ? | GWh |
| **Erreur pointe** | A-t-on bien prévu le maximum de la journée ? | MW |
| **Écart heure de pointe** | A-t-on trouvé la bonne heure du maximum ? | heures |

**Les mêmes jours pour tout le monde :** le sujet exige de comparer les méthodes **sur les
mêmes dates**. Un jour où une méthode n'a pas ses 24 valeurs est retiré pour **toutes** les
méthodes.

**Les jours de changement d'heure** (2 par an) sont retirés de la notation : ils n'ont pas
24 heures, on ne peut pas les comparer heure par heure (piste de la décision 11).

In [ ]:
voir_code(evaluation.erreurs_par_jour)

## 5. Les résultats

Je calcule les deux benchmarks sur l'**apprentissage** (2016-2022), pour information, et sur la
**validation** (2023). C'est la validation qui servira de référence pour choisir les modèles.

In [ ]:
scores_validation, reel, previsions = benchmarks.evaluer_periode("validation", conso_h)
scores_apprentissage, _, _ = benchmarks.evaluer_periode("apprentissage", conso_h)

print("=== Validation 2023 ===")
scores_validation.round(1)

In [ ]:
print("=== Apprentissage 2016-2022 (pour information) ===")
scores_apprentissage.round(1)

**Ce que je lis :**
- **B1 est le meilleur des deux** : environ **3 460 MW** d'erreur moyenne en 2023, soit
  **6,5 %** de la consommation. B2 fait nettement moins bien (environ 4 170 MW, 7,9 %).
- Les résultats sont **très proches entre l'apprentissage et la validation** : les benchmarks
  ont un niveau d'erreur stable d'une année à l'autre.
- **Le score à battre est donc celui de B1 : environ 3 460 MW de MAE sur 2023.**
- Le biais est presque nul sur l'année : les benchmarks ne se trompent pas toujours dans le
  même sens. Ils se trompent dans un sens l'hiver et dans l'autre au printemps (voir plus bas).
- 362 jours sur 365 : les 2 jours de changement d'heure, plus un jour où B1 n'a pas ses
  24 heures (il recopie le dimanche de 23 h).

## 6. Où les benchmarks se trompent, et ce que ça nous apprend

Les erreurs des benchmarks montrent ce que les modèles devront mieux faire.

In [ ]:
jours = evaluation.jours_comparables(reel, *previsions.values())
erreurs = {nom: evaluation.erreurs_par_jour(reel.loc[jours], p.loc[jours]) for nom, p in previsions.items()}
e_b1, e_b2 = erreurs["B1 : semaine précédente"], erreurs["B2 : moyenne de 4 semaines"]

# Erreur moyenne par mois
par_mois = pd.DataFrame({
    "B1": e_b1["mae"].groupby(e_b1.index.month).mean(),
    "B2": e_b2["mae"].groupby(e_b2.index.month).mean(),
}) / 1000

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(12)
ax.bar(x - 0.2, par_mois["B1"], width=0.38, color=BLEU, label="B1 : semaine précédente")
ax.bar(x + 0.2, par_mois["B2"], width=0.38, color=ORANGE, label="B2 : moyenne de 4 semaines")
ax.set_xticks(x, ["janv", "févr", "mars", "avr", "mai", "juin",
                  "juil", "août", "sept", "oct", "nov", "déc"])
ax.set_ylabel("MAE (GW)")
ax.set_title("Erreur moyenne des benchmarks par mois, validation 2023")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False)
plt.show()

**Ce que je lis :**
- **L'hiver est beaucoup plus difficile** : environ 6 à 7 GW d'erreur en janvier, février et
  décembre, contre moins de 1 GW en juin. En hiver, la consommation dépend du **froid**, qui
  change d'une semaine à l'autre. Les benchmarks ne connaissent pas la température : c'est
  là que la **météo** devrait apporter le plus.
- **B2 est en retard sur les saisons.** En novembre, quand il fait de plus en plus froid, la
  moyenne des 4 semaines précédentes regarde des semaines plus douces : B2 prévoit trop bas
  (biais d'environ −6 500 MW). Au printemps, c'est l'inverse (il prévoit trop haut). B1, qui
  ne regarde qu'une semaine en arrière, suit mieux. C'est pour ça que B1 gagne.

In [ ]:
# Erreur moyenne selon l'heure de la journée prévue
ecart_heure = pd.DataFrame({
    nom: (p.loc[jours] - reel.loc[jours]).abs().mean() / 1000 for nom, p in previsions.items()
})
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(ecart_heure.index, ecart_heure.iloc[:, 0], color=BLEU, marker="o", markersize=4,
        label="B1 : semaine précédente")
ax.plot(ecart_heure.index, ecart_heure.iloc[:, 1], color=ORANGE, marker="o", markersize=4,
        label="B2 : moyenne de 4 semaines")
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Heure prévue de J+1 (Paris)")
ax.set_ylabel("MAE (GW)")
ax.set_ylim(bottom=0)
ax.set_title("Erreur moyenne selon l'heure prévue, validation 2023")
ax.legend(frameon=False, loc="lower right")
plt.show()

**Ce que je lis :** les erreurs sont les plus grandes **le matin, vers 7-9 h**, au moment où
la consommation monte vite (réveil, chauffage), avec une deuxième bosse vers **18-19 h**, à la
pointe du soir. La nuit est plus facile. B1 est meilleur que B2 à toutes les heures.

In [ ]:
# Les 10 jours où B1 s'est le plus trompé en 2023
pires = e_b1.sort_values("mae", ascending=False).head(10)
pd.DataFrame({
    "jour": pires.index.strftime("%A %d/%m/%Y"),
    "MAE (MW)": pires["mae"].round(0).astype(int),
    "biais (MW)": pires["biais"].round(0).astype(int),
}).reset_index(drop=True)

**Ce que je lis :** les plus grosses erreurs ont deux causes, qui correspondent exactement
aux deux informations que les modèles auront en plus :

1. **Le calendrier :** le **lundi 25 décembre** (Noël) est la pire journée. B1 recopie le lundi
   18 décembre, un jour de travail normal, et prévoit environ 19 000 MW de trop (biais positif).
   Même chose le mardi 26. Il faudra des variables **jours fériés** et **période de Noël**.
2. **La météo :** du **jeudi 19 au lundi 23 janvier**, puis les **27-28 février**, une vague de
   froid arrive. B1 recopie une semaine plus douce et prévoit trop bas (biais négatif). Le
   **lundi 13 mars**, c'est l'inverse : la semaine précédente était plus froide. Il faudra la
   **température**.

Ces journées sont de bonnes candidates pour l'**analyse des échecs** demandée par le sujet
(au moins trois journées expliquées). Il faudra les reprendre avec les modèles.

In [ ]:
print(f"B1 fait mieux que B2 {(e_b1['mae'] < e_b2['mae']).mean():.0%} des jours en 2023.")
print("Erreur moyenne de B1 par année (apprentissage) :")
_, reel_app, prev_app = benchmarks.evaluer_periode("apprentissage", conso_h)
jours_app = evaluation.jours_comparables(reel_app, *prev_app.values())
e_app = evaluation.erreurs_par_jour(reel_app.loc[jours_app], prev_app["B1 : semaine précédente"].loc[jours_app])
(e_app["mae"].groupby(e_app.index.year).mean()).round(0).astype(int).rename("MAE B1 (MW)")

## Résumé

| Question | Réponse |
|---|---|
| Meilleur benchmark | **B1** (semaine précédente), il gagne environ 63 % des jours |
| **Score à battre (validation 2023)** | **MAE ≈ 3 460 MW (6,5 %)**, RMSE ≈ 5 000 MW |
| Stabilité | Erreur de B1 entre environ 3 100 et 4 100 MW selon les années |
| Quand les benchmarks échouent | En **hiver** (le froid change), aux **jours fériés** (Noël), le **matin** |
| Ce que les modèles devront apporter | La **température** et le **calendrier** (fériés, Noël) |
| Règle des 14 h | Vérifiée pour chaque jour : aucune triche |
| Test 2024-2025 | **Pas utilisé**, protégé par le code |

**Prochaine étape :** le premier modèle appris (M1 : régression avec le calendrier et les
retards de consommation), qui devra battre B1 sur 2023.